# Spike 02 — Arabic OCR with Gemini 2.5 Flash

**Goal:** Extract Arabic text from the Tranquil City page images using Gemini 2.5 Flash.

**Time box:** 1 hour

**Question to answer:** Does Gemini produce accurate, well-structured Arabic Markdown from these report pages?

**Done when:** `.md` files in `ocr_output/arabic/` contain readable, correctly-structured Arabic text that a native speaker confirms is accurate.

---

### Why Gemini 2.5 Flash for Arabic?

From the Chandra full benchmark (the only two models compared):

| Model | Arabic Score |
|---|---|
| **Gemini 2.5 Flash** | **84.4%** ✅ |
| Chandra 2 | 68.4% |

16 percentage point gap. For Arabic — with its connected script, optional diacritics, and RTL layout — this difference is significant enough to matter in a production RAG system.

### API Key Setup
1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with a **new Google account** (to get the \$300 free credit)
3. Create API key → copy it
4. Paste it in your `.env` file: `GEMINI_API_KEY=your_key_here`

## Step 1 — Imports & Setup

In [1]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
from pathlib import Path
import os
import time

load_dotenv(dotenv_path="../.env")

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env file")

# New SDK: client-based instead of module-level configure()
client = genai.Client(api_key=api_key)
MODEL  = "gemini-2.5-flash"

print("✅ Gemini client ready  (google-genai SDK)")
print(f"   Model: {MODEL}")

✅ Gemini client ready  (google-genai SDK)
   Model: gemini-2.5-flash


## Step 2 — The OCR Prompt

The prompt is the most important variable here. A vague prompt gives vague output.

We tell Gemini exactly what we want:
- Extract ALL text (nothing skipped)
- Preserve structure (headings, tables, lists)
- Output in Markdown (machine-readable for RAG)
- No commentary (just the text)

In [2]:
OCR_PROMPT = """\
أنت محرك OCR متخصص. مهمتك استخراج كامل النص من صورة الوثيقة.

القواعد:
- استخرج النص العربي بشكل كامل ودقيق مع الحفاظ على الحركات إن وجدت
- حافظ على بنية الجداول باستخدام صيغة Markdown للجداول
- حافظ على تسلسل العناوين (# للعنوان الرئيسي، ## للعناوين الفرعية)
- حافظ على الأرقام كما هي بالضبط
- احتفظ بالنص الإنجليزي كما هو إذا ظهر في الصفحة
- لا تضف أي تعليق أو شرح — فقط النص المستخرج

You are a specialized OCR engine. Extract ALL text from this document image.
Rules:
- Extract Arabic text accurately, preserving diacritics if present
- Preserve table structure using Markdown table format
- Preserve heading hierarchy (# for main titles, ## for sections)
- Preserve numbers exactly as shown
- Keep any English text as-is
- Output ONLY the extracted text — no commentary, no explanation
"""

print("✅ OCR prompt defined")
print(f"   Prompt length: {len(OCR_PROMPT)} characters")

✅ OCR prompt defined
   Prompt length: 791 characters


## Step 3 — OCR a Single Page First (validate before bulk run)

Always test on ONE page before running the whole document.  
This saves API credits and catches prompt issues early.

In [6]:
def ocr_page(image_path: str) -> str:
    """
    Sends a page image to Gemini and returns extracted Markdown text.

    New google-genai SDK uses client.models.generate_content()
    with typed Part objects instead of raw dicts.
    """
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    response = client.models.generate_content(
        model    = MODEL,
        contents = [
            types.Part.from_text(text=OCR_PROMPT),
            types.Part.from_bytes(data=image_bytes, mime_type="image/png")
        ]
    )
    return response.text


# Test on page 6 (more likely to have real content than the cover page)
ar_images = sorted(Path("../data/images_ar_subset").glob("*.png"))

if not ar_images:
    print("❌ No Arabic images found — run Spike 01 first")
else:
    print(f"Found {len(ar_images)} Arabic page images")

    test_page = ar_images[15] if len(ar_images) > 5 else ar_images[0]
    print(f"Testing OCR on: {test_page.name}")
    print()

    test_result = ocr_page(str(test_page))

    print("=== OCR OUTPUT (first 1000 chars) ===")
    print(test_result[:1000])
    print()
    print(f"Total characters extracted: {len(test_result)}")

Found 29 Arabic page images
Testing OCR on: page_038.png

=== OCR OUTPUT (first 1000 chars) ===
# مشروع حافلات المدينة

يعد مشروع حافلات المدينة من المشاريع الهامة التي تهدف إلى خدمة نقل سكان وزوار المدينة
المنورة من وإلى عدة نقاط رئيسية، أبرزها المسجد النبوي الشريف ومسجد قباء ومطار الأمير
محمد بن عبد العزيز ومحطة قطار الحرمين، وعدد من المراكز الحيوية مثل (جادة قباء وسيد
الشهداء وعدد من المستشفيات والمجمعات التجارية)، وذلك من خلال 6 مسارات تمر بـ 123
محطة، بإجمالي طول شبكة تتعدى 232.3 كم، حيث تربط شمال المدينة بجنوبها وشرقها
بغربها مروراً بالمسجد النبوي الشريف.

## خريطة حافلات المدينة المنورة

| من                      | إلى                          |
| :---------------------- | :--------------------------- |
| مسار 300 قطار الحرمين السريع      | المسجد النبوي الشريف         |
| مسار 400 مطار الأمير محمد بن عبد العزيز | المسجد النبوي الشريف         |
| مسار 401 العالية              | الجامعة الإسلامية          |
| مسار 402 الخالدية             | الميقات                     |
| مسار 40

## Step 4 — Human Check Before Continuing

⚠️ **Stop here.** Before running OCR on all pages, answer these questions:

1. Is the Arabic text readable and correct? (Have a native speaker verify)
2. Are tables formatted properly as Markdown tables?
3. Are numbers preserved exactly?

If any answer is NO → adjust the `OCR_PROMPT` above and re-run Step 3.  
If all YES → proceed to Step 5.

In [7]:
# Quick self-check — does the output look like Arabic Markdown?
if 'test_result' in dir():
    has_arabic = any('\u0600' <= c <= '\u06ff' for c in test_result)
    has_content = len(test_result.strip()) > 50

    print("Quick validation:")
    print(f"  Contains Arabic characters : {'✅ Yes' if has_arabic else '❌ No'}")
    print(f"  Has substantial content    : {'✅ Yes' if has_content else '❌ No — likely empty or error'}")
    print(f"  Has Markdown headers       : {'✅ Yes' if '#' in test_result else '⚠️  No headers found'}")
    print(f"  Has Markdown tables        : {'✅ Yes' if '|' in test_result else '⚠️  No tables found (may be OK if page has no tables)'}")

Quick validation:
  Contains Arabic characters : ✅ Yes
  Has substantial content    : ✅ Yes
  Has Markdown headers       : ✅ Yes
  Has Markdown tables        : ✅ Yes


## Step 5 — Run OCR on All Pages

**IMPORTANT:** Gemini free tier = 15 requests/minute.  
We add a small delay between calls to stay within the limit.  
With \$300 credits on a new account, rate limits are much higher.

In [8]:
OUTPUT_DIR = Path("../ocr_output/arabic")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DELAY_BETWEEN_CALLS = 4  # seconds — keeps us under 15 req/min on free tier

results = {}  # page_name → extracted text

for i, img_path in enumerate(ar_images):
    out_path = OUTPUT_DIR / img_path.with_suffix(".md").name

    # Skip if already processed (allows resuming interrupted runs)
    if out_path.exists():
        print(f"⏭  Skipping {img_path.name} — already processed")
        results[img_path.stem] = out_path.read_text(encoding="utf-8")
        continue

    print(f"[{i+1}/{len(ar_images)}] Processing {img_path.name}...", end=" ")

    try:
        text = ocr_page(str(img_path))
        out_path.write_text(text, encoding="utf-8")
        results[img_path.stem] = text
        print(f"✅ {len(text)} chars saved to {out_path.name}")
    except Exception as e:
        print(f"❌ Error: {e}")

    # Respect rate limit — skip delay on the last page
    if i < len(ar_images) - 1:
        time.sleep(DELAY_BETWEEN_CALLS)

print(f"\n✅ Done — {len(results)} pages processed")

[1/29] Processing page_007.png... ✅ 1485 chars saved to page_007.md
[2/29] Processing page_008.png... ✅ 528 chars saved to page_008.md
[3/29] Processing page_010.png... ✅ 2785 chars saved to page_010.md
[4/29] Processing page_011.png... ✅ 3019 chars saved to page_011.md
[5/29] Processing page_013.png... ❌ Error: [Errno 2] No such file or directory: '..\\data\\images_ar_subset\\page_013.png'
[6/29] Processing page_015.png... ❌ Error: [Errno 2] No such file or directory: '..\\data\\images_ar_subset\\page_015.png'
[7/29] Processing page_017.png... ❌ Error: [Errno 2] No such file or directory: '..\\data\\images_ar_subset\\page_017.png'
[8/29] Processing page_020.png... ✅ 798 chars saved to page_020.md
[9/29] Processing page_021.png... ❌ Error: [Errno 2] No such file or directory: '..\\data\\images_ar_subset\\page_021.png'
[10/29] Processing page_023.png... ❌ Error: [Errno 2] No such file or directory: '..\\data\\images_ar_subset\\page_023.png'
[11/29] Processing page_025.png... ❌ Error: [E

## Step 6 — Inspect Any Page's Output

In [9]:
# Change page number to inspect different pages
PAGE_TO_INSPECT = 2

md_files = sorted(OUTPUT_DIR.glob("*.md"))
if len(md_files) >= PAGE_TO_INSPECT:
    chosen = md_files[PAGE_TO_INSPECT - 1]
    content = chosen.read_text(encoding="utf-8")
    print(f"File: {chosen.name}")
    print(f"Size: {len(content)} characters")
    print("=" * 60)
    print(content)
else:
    print(f"Only {len(md_files)} pages processed so far")

File: page_008.md
Size: 528 characters
# المحتوى

## المقدمة
3

## الفصل الأول
المنظور المكاني والتاريخي للمدينة المنورة
5

## الفصل الثاني
مدينة الأنسنة القابلة للعيش
13

## الفصل الثالث
التنقل الحضري
27

## الفصل الرابع
الإسكان اللائق ميسور التكلفة
37

## الفصل الخامس
الاستدامة البيئية
43

## الفصل السادس
جودة الرعاية الصحية
49

## الفصل السابع
جودة التعليم
59

## الفصل الثامن
رأس المال الاجتماعي والترابط والاندماج الاجتماعي
67

## الفصل التاسع
اقتصاد مزدهر وحيوي
71

## الفصل العاشر
السياحة والترفيه
77

## الفصل الحادي عشر
مؤشر قابلية العيش للمدينة المنورة
93


## Step 7 — Spike Summary

In [10]:
md_files = sorted(OUTPUT_DIR.glob("*.md"))
total_chars = sum(f.read_text(encoding="utf-8").__len__() for f in md_files)

print("=" * 50)
print("SPIKE 02 — RESULTS")
print("=" * 50)
print(f"Pages processed     : {len(md_files)} / {len(ar_images)}")
print(f"Total chars extracted: {total_chars:,}")
print(f"Avg chars per page  : {total_chars // max(len(md_files),1):,}")
print()

if len(md_files) == len(ar_images) and total_chars > 1000:
    print("✅ SPIKE PASSED — Arabic OCR output ready for Spike 04 (PageIndex)")
    print(f"   Files in: {OUTPUT_DIR.resolve()}")
else:
    print("⚠️  SPIKE INCOMPLETE")
    print(f"   {len(ar_images) - len(md_files)} pages still need processing")

SPIKE 02 — RESULTS
Pages processed     : 14 / 29
Total chars extracted: 24,228
Avg chars per page  : 1,730

⚠️  SPIKE INCOMPLETE
   15 pages still need processing
